Import libraries

In [ ]:
from lxml import html, etree
import pandas as pd
import pyperclip

Declare namespaces

In [ ]:
ns = {'svg': 'http://www.w3.org/2000/svg'}

Process connections table with navgraph svg in connections html

In [ ]:
def draw_connections(col):
    myHTML = html.parse('connections.html')
    nodeGroup = myHTML.xpath('//g[@id="nodes"]')[0]
    #print(nodeGroup)
    navgraphNodeIds = [node.get('id') for node in nodeGroup.findall('g', ns)]
    #print(f'Node IDs: {navgraphNodeIds}')
    connectionGroup = etree.fromstring(f'<g class="connections" id="{col}" display="none"/>')
    
    connectionDf = pd.read_csv('https://docs.google.com/spreadsheets/d/15mjhEAJDR5rhdux3mdjVVLWATm20uo_droIn5K4kRXA/export?format=csv&gid=1532912149')
    
    # print(connectionDf)
    for index, row in connectionDf.iterrows():
        if pd.isna(row[col]):
            continue
        myConnection = row[col].strip()
        myFromId = row['from']
        if myFromId not in navgraphNodeIds:
            print(f'!!!{myFromId} not in Navgraph!!!')
            continue
        myToId = row['to']
        if myToId not in navgraphNodeIds:
            print(f'!!!{myToId} not in Navgraph!!!')
            continue
        
        myClass = ''
        if 'connected' in myConnection and 'label' in myConnection:
            myClass = 'connected_label'
        elif 'line' in myConnection and 'label' in myConnection:
            myClass = 'line_label'
        elif 'connected' in myConnection:
            myClass = 'connected'
        elif 'line' in myConnection:
            myClass = 'line'
        elif 'label' in myConnection:
            myClass = 'label'
        
        # print(f'From {myFromId} to {myToId}:')
        myNodeFrom = nodeGroup.find(f'g[@id="{myFromId}"]', ns)
        cx_from = myNodeFrom.find('circle', ns).get('cx')
        cy_from = myNodeFrom.find('circle', ns).get('cy')

        myNodeTo = nodeGroup.find(f'g[@id="{myToId}"]', ns)
        cx_to = myNodeTo.find('circle', ns).get('cx')
        cy_to = myNodeTo.find('circle', ns).get('cy')

        myLine = etree.fromstring(f'<line x1="{cx_from}" y1="{cy_from}" x2="{cx_to}" y2="{cy_to}" class="{myClass}" stroke-width="3"/>')
        connectionGroup.append(myLine)
        # print(f'Line: {lineString}')
    outputGroup = etree.tostring(connectionGroup, encoding='unicode', pretty_print=True)
    print(outputGroup)
    pyperclip.copy(outputGroup) #copy to clipboard

execute function

In [ ]:
draw_connections('F1')